## Reproduction with Authors' Code

written by **Jinwoo Lee**.    
jil527@ucsd.edu | jinwoo-lee.com

Dec, 2025   
as a PSYC201A's **Reproducibility Project**     

**Note:** This script aims to reproduce the key findings of Kim & Kim (2022) with the authors' original code and data.      

This notebook assumes that you have already obtained the data from the authors and placed them in the appropriate directories as specified in the README file. The original code is available at: https://github.com/humaffectneurolab/MorphSim_TraitAnxiety/blob/main/morphsim.R. 

---
### Step 1. Loading the Packages

In [1]:
library(vegan)
library(reshape)

Loading required package: permute



---
### Step 2. Constructing an Anxiety Distance Matrix

The authors selected a total of **119** and **45** participants for the **youth** and **older** adults group, respecitvely. To filter the sample, the authors used following three criteria; 1) age (20-35 and 60-75 years for youth and older group, respectively), 2) MRI availability, and 3) Absence of psychiatric disorders (using SKID diagnosis).      

For each group, the authors constructed an intersubject matrix of mean trait anxiety score using STAI-G-X2 measures - `Hyouth_anx` and `Hsenior_anx`.

In [2]:
meta = read.csv("../data-from-authors/Meta.csv")
STAI = read.csv("../data-from-authors/STAI_G_X2.csv")

### ==================================================================
### PART I. YOUNG ADULTS GROUP 
# age
youth = subset(meta, Age == "20-25" | Age == "25-30" | Age == "30-35")

# IDP availability
youth = subset(youth, X != "sub-032339" & X != "sub-032341" & X != "sub-032459" & X != "sub-032370"
               & X != "sub-032466" & X != "sub-032438" & X != "sub-032509")

# SKID Diagnosis
youth$SKID_Diagnoses = as.factor(youth$SKID_Diagnoses)
youth$SKID_Diagnoses = as.numeric(youth$SKID_Diagnoses)
Hyouth = subset(youth, SKID_Diagnoses == 1 | SKID_Diagnoses == 10)

# leaving only relevant info
Hyouth = Hyouth[, -c(4:14, 16:21)]

# merging with STAI
Hyouth = merge(Hyouth, STAI, by = 'X')

# construct Mean Anxiety Matrix
Hyouth_anx = data.frame()

for (i in 1:119) {
  for (j in 1:119) {
    Hyouth_anx[j, i] = (sum(Hyouth[i, 5], Hyouth[j, 5])) / 2
  }
}

### ==================================================================
### PART II. OLDER ADULTS GROUP 
# age
senior = subset(meta, Age == "60-65" | Age == "65-70" | Age == "70-75")

# SKID Diagnosis
senior$SKID_Diagnoses = as.factor(senior$SKID_Diagnoses)
senior$SKID_Diagnoses = as.numeric(senior$SKID_Diagnoses)
Hsenior = subset(senior, SKID_Diagnoses == 1 | SKID_Diagnoses == 5)

# IDP availability
Hsenior = subset(Hsenior, X != "sub-032339" & X != "sub-032341" & X != "sub-032459" & X != "sub-032370" 
            & X != "sub-032466" & X != "sub-032438" & X != "sub-032509" & X != "sub-032392" 
            & X != "sub-032443" & X != "sub-032488")

# leaving only relevant info
Hsenior = Hsenior[, -c(4:14, 16:21)]

# merging with STAI
Hsenior = merge(Hsenior, STAI, by = 'X')

# construct Mean Anxiety Matrix
Hsenior_anx = data.frame()

for (i in 1:45) {
  for (j in 1:45) {
    Hsenior_anx[j, i] = (sum(Hsenior[i, 5], Hsenior[j, 5])) / 2
  }
}

---
### Step 3. Constructing a Brain Distance Matrix

For each participant, the authors computed an one-dimensional vector which represents each participant’s left vPFC–amygdala morphology using voxel-level streamline count features that had been preprocessed with a **5%** slice-level thresholding procedure (see Fig 1A in the original paper).       


Using participants' vector data, the authors construced an intersubject matrix of Euclidean distance for each group - `Hyouth_brain` and `Hsenior_brain`. 


In [3]:
### ======================================================================================
### I. YOUNG ADULT GROUP 
yh_prob_5p_L = read.csv("../data-from-authors/tractfiles/yh_probmap_5p_L.csv", header = T)
yh_prob_5p_L = yh_prob_5p_L[,-1]  # removing the first column which is just the index

Hyouth_brain = dist(t(yh_prob_5p_L))

### =======================================================================================
### II. OLDER ADULT GROUP 
oh_prob_5p_L = read.csv("../data-from-authors/tractfiles/oh_prob_5p_L_new.csv", header = T)
oh_prob_5p_L = oh_prob_5p_L[,-1]  # removing the first column which is just the index

Hsenior_brain = dist(t(oh_prob_5p_L))

---
### Step 4. Anna Karenina Testing

Using `vegan::mantel` function, the Spearman's $\rho$ between two intersubject matrices - `_anx` and `_brain`. Afterward, its statistical significance was evaluated through 10,000 iterations of the Mantel permutation test. Since the original code did not fix a random seed, we arbitrarily set the seed to 42 for our analysis.

In [4]:
### YOUNG ADULTS GROUP
set.seed(42)
mantel(Hyouth_anx, Hyouth_brain, method = "spearman", permutations = 10000)


Mantel statistic based on Spearman's rank correlation rho 

Call:
mantel(xdis = Hyouth_anx, ydis = Hyouth_brain, method = "spearman",      permutations = 10000) 

Mantel statistic r: 0.1485 
      Significance: 0.041596 

Upper quantiles of permutations (null model):
  90%   95% 97.5%   99% 
0.109 0.140 0.169 0.196 
Permutation: free
Number of permutations: 10000


In [5]:
### SENIOR ADULTS GROUP
set.seed(42)
mantel(Hsenior_anx, Hsenior_brain, method = "spearman", permutations = 10000)


Mantel statistic based on Spearman's rank correlation rho 

Call:
mantel(xdis = Hsenior_anx, ydis = Hsenior_brain, method = "spearman",      permutations = 10000) 

Mantel statistic r: 0.2945 
      Significance: 0.016098 

Upper quantiles of permutations (null model):
  90%   95% 97.5%   99% 
0.183 0.234 0.272 0.319 
Permutation: free
Number of permutations: 10000


For visualizing the scatterplot between mean anxiety score and brain morphological dissimilarity value, we saved a dataframe for each group - `df_young` and `df_older`. Each dataframe has columns corresponding to the vectors obtained by flattening the upper-triangular elements of the diagonal of the anxiety and brain structure distance matrices. The rows represent all possible pairwise combinations of participants within each group.

In [6]:
### Dataframe for young group
Hyouth_brain <- as.data.frame(as.matrix(Hyouth_brain))

Hyouth_anx_vec <- Hyouth_anx[upper.tri(Hyouth_anx)]
Hyouth_brain_vec <- Hyouth_brain[upper.tri(Hyouth_brain)]

df_youth <- data.frame(anx_mean = Hyouth_anx_vec, brain_dissimilarity = Hyouth_brain_vec)

### Dataframe for older group
Hsenior_brain <- as.data.frame(as.matrix(Hsenior_brain))
Hsenior_anx_vec <- Hsenior_anx[upper.tri(Hsenior_anx)]
Hsenior_brain_vec <- Hsenior_brain[upper.tri(Hsenior_brain)]

df_older <- data.frame(anx_mean = Hsenior_anx_vec, brain_dissimilarity = Hsenior_brain_vec)

### Save the dataframe
write.csv(df_youth, "results/confirmatory/youth_vectors_authors.csv", row.names = FALSE)
write.csv(df_older, "results/confirmatory/older_vectors_authors.csv", row.names = FALSE)